In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

HAZARD_CSV = Path("data") / "hazard.csv"
EXPOSURE_CSV = Path("data") / "exposure_district.csv"
VULNERABILITY_CSV = Path("data") / "vulnerability.csv"
GOVT_CSV = Path("data") / "government_response_district.csv"
MASTER_CSV = Path("data") / "MASTER_VARIABLES.csv"

hazard = pd.read_csv(HAZARD_CSV)
exposure = pd.read_csv(EXPOSURE_CSV)
vulnerability = pd.read_csv(VULNERABILITY_CSV)
gov = pd.read_csv(GOVT_CSV)

# =============================================================================
# STANDARDIZE COLUMN NAMES
# (Replace "_" with "-" only in headers)
# =============================================================================

hazard.columns = hazard.columns.str.replace("_", "-", regex=False)
exposure.columns = exposure.columns.str.replace("_", "-", regex=False)
vulnerability.columns = vulnerability.columns.str.replace("_", "-", regex=False)
gov.columns = gov.columns.str.replace("_", "-", regex=False)

# =============================================================================
# KEEP REQUIRED COLUMNS
# =============================================================================

hazard = hazard[["district", "timeperiod", "heat-hazard", "heat-days-score"]]
exposure = exposure[["district", "timeperiod", "exposure"]]
vulnerability = vulnerability[["district", "timeperiod", "vulnerability"]]
gov = gov[["district", "timeperiod", "government-response"]]

# =============================================================================
# MERGE COMPONENTS (DISTRICT × TIMEPERIOD)
# =============================================================================

df = hazard.merge(exposure, on=["district", "timeperiod"], how="inner")
df = df.merge(vulnerability, on=["district", "timeperiod"], how="inner")
df = df.merge(gov, on=["district", "timeperiod"], how="inner")

# =============================================================================
# STANDARDIZE MERGE KEYS
# =============================================================================

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# WEIGHTS
# =============================================================================

weights = {
    "heat-hazard": 4,
    "exposure": 1,
    "vulnerability": 2,
    "government-response": 2,
}

total_weight = sum(weights.values())

# =============================================================================
# TOPSIS PER TIMEPERIOD
# =============================================================================

results = []

for tp, g in df.groupby("timeperiod"):
    g = g.copy()

    # -------------------------------------------------------------------------
    # Min-Max Normalization
    # -------------------------------------------------------------------------
    norm = pd.DataFrame(index=g.index)

    for col in weights:
        min_v = g[col].min()
        max_v = g[col].max()

        if max_v == min_v:
            norm[col] = 0
        else:
            norm[col] = (g[col] - min_v) / (max_v - min_v)

    # -------------------------------------------------------------------------
    # Apply weights
    # -------------------------------------------------------------------------
    for col in weights:
        norm[col] *= weights[col] / total_weight

    # -------------------------------------------------------------------------
    # Ideal Best / Worst
    # -------------------------------------------------------------------------
    ideal_best = norm.max()
    ideal_worst = norm.min()

    # -------------------------------------------------------------------------
    # Euclidean Distances
    # -------------------------------------------------------------------------
    dist_best = np.sqrt(((norm - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((norm - ideal_worst) ** 2).sum(axis=1))

    # -------------------------------------------------------------------------
    # TOPSIS Score
    # -------------------------------------------------------------------------
    g["topsis-score"] = dist_worst / (dist_best + dist_worst)

    results.append(g)

# =============================================================================
# COMBINE RESULTS
# =============================================================================

df = pd.concat(results, ignore_index=True)

# =============================================================================
# RISK CLASSIFICATION
# =============================================================================

def classify(score):
    if score <= 0.2:
        return 1
    elif score <= 0.4:
        return 2
    elif score <= 0.6:
        return 3
    elif score <= 0.8:
        return 4
    else:
        return 5


df["heat-risk-score"] = df["topsis-score"].apply(classify)

# =============================================================================
# SUMMARY
# =============================================================================

print("\nTOPSIS Summary:")
print(df["topsis-score"].describe())

print("\nRisk Class Distribution:")
print(df["heat-risk-score"].value_counts().sort_index())

print("\nPreview:")
print(df[["district", "timeperiod", "topsis-score", "heat-risk-score"]].head())

# =============================================================================
# APPEND RESULTS TO MASTER_VARIABLES
# =============================================================================

master = pd.read_csv(MASTER_CSV)
fy_cumsum = pd.read_csv("data/government_response_district.csv")

# Standardize headers in MASTER_VARIABLES too
master.columns = master.columns.str.replace("_", "-", regex=False)
fy_cumsum.columns = fy_cumsum.columns.str.replace("_", "-", regex=False)

master["district"] = master["district"].astype(str).str.strip()
fy_cumsum["district"] = fy_cumsum["district"].astype(str).str.strip()
master["timeperiod"] = master["timeperiod"].astype(str).str.strip()
fy_cumsum["timeperiod"] = fy_cumsum["timeperiod"].astype(str).str.strip()

# Remove old columns if present
for col in [
    "heat-hazard",
    "exposure",
    "vulnerability",
    "government-response",
    "topsis-score",
    "heat-risk-score",
]:
    if col in master.columns:
        master = master.drop(columns=col)

# Merge scores back
final_df = master.merge(
    df[
        [
            "district",
            "timeperiod",
            "heat-hazard",
            "exposure",
            "vulnerability",
            "government-response",
            "topsis-score",
            "heat-risk-score",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
)

final_df = final_df.merge(
    fy_cumsum[
        [
            "district",
            "timeperiod",
            "cum-tender-value",

        ]
    ],
    on=["district", "timeperiod"],
    how="left",
)

# =============================================================================
# SAVE FINAL OUTPUT
# =============================================================================

OUTPUT = Path("data") / "archive" / "final_risk_score.csv"

final_df.to_csv(OUTPUT, index=False)

print(f"\nSaved final file: {OUTPUT}")
print(f"Rows: {len(final_df)}")
print(f"Columns: {len(final_df.columns)}")


# =============================================================================
# DISTRICT-LEVEL FINAL RISK SCORE
# =============================================================================

district_df = (
    final_df.groupby(["district", "timeperiod"], as_index=False)
    .agg(
        {
            # Identifiers
            "dtname": "first",

            # Sum
            "HealthCenters": "sum",

            # Means
            "cum-tender-value": "first",
            "total-tender-awarded-value": "sum",
            "mean-heatday": "mean",
            "sum-aged-population": "sum",
            "sum-young-population": "sum",
            "sum-population": "sum",
            "avg-electricity": "mean",
            "block-piped-hhds-pct": "mean",
            "block-nosanitation-hhds-pct": "mean",
            "total-hhd": "sum",
            "nco-5-9-percent-estimated": "mean",
            "women-sugar": "mean",
            "men-sugar": "mean",
            "women-bp": "mean",
            "men-bp": "mean",
            "pct-ncd": "mean",
            "land-surface-temperature-raster": "first",
            "heat-hazard": "first",
            "exposure": "first",
            "vulnerability": "first",
            "government-response": "first",
            "topsis-score": "first",
            "heat-risk-score": "first",
            
            # lst classes
            "land-surface-temperature": "mean",
        }
    )
)

# Round numeric columns
numeric_cols = district_df.select_dtypes(include=np.number).columns
district_df[numeric_cols] = district_df[numeric_cols].round(3)

# Create district -> object_id lookup
master_df = pd.read_csv(MASTER_CSV)
object_lookup = (
    master_df[["district", "object_id"]]
    .drop_duplicates(subset="district")
)
# Add object_id to district_df
district_df = district_df.merge(
    object_lookup,
    on="district",
    how="left"
)

# Keep only the first two parts of the object_id to represent only the district
district_df["object_id"] = (
    district_df["object_id"]
    .astype(str)
    .str.rsplit("-", n=1)
    .str[0]
)

# Rename columns
district_df = district_df.rename(
    columns={
        "HealthCenters": "health-centres-count",
        "block-piped-hhds-pct": "piped-hhds-pct",
        "block-nosanitation-hhds-pct": "nosanitation-hhds-pct",
        "mean-heatday": "heat-days-score",
        "nco-5-9-percent-estimated": "workers-affected-pct",
        "object_id": "object-id",
        "cum-tender-value": "total-tender-awarded-value-fy-cumsum"
    }
)

# =============================================================================
# SAVE
# =============================================================================


DISTRICT_OUTPUT = Path("data") / "district_final_risk_score.csv"

district_df.to_csv(DISTRICT_OUTPUT, index=False)

print(f"\nSaved district-level file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================




DISTRICT_OUTPUT = Path("data") / "district_final_risk_score.csv"

# rounding off
# Population columns as integers
int_cols = [
    "sum-aged-population",
    "sum-young-population",
    "sum-population",
]

district_df[int_cols] = district_df[int_cols].round().astype("Int64")

# All other numeric columns to 2 decimal places
numeric_cols = district_df.select_dtypes(include="number").columns
decimal_cols = numeric_cols.difference(int_cols)

district_df[decimal_cols] = district_df[decimal_cols].round(2)


district_df.to_csv(DISTRICT_OUTPUT, index=False)

print(f"\nSaved district-level file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")


TOPSIS Summary:
count    1980.000000
mean        0.455363
std         0.162632
min         0.000000
25%         0.338365
50%         0.454339
75%         0.578835
max         0.951979
Name: topsis-score, dtype: float64

Risk Class Distribution:
heat-risk-score
1    110
2    604
3    845
4    397
5     24
Name: count, dtype: int64

Preview:
    district timeperiod  topsis-score  heat-risk-score
0     Anugul    2021_01      0.578835                3
1   Balangir    2021_01      0.417840                3
2  Baleshwar    2021_01      0.285855                2
3    Bargarh    2021_01      0.218686                2
4    Bhadrak    2021_01      0.353889                2

Saved final file: data/archive/final_risk_score.csv
Rows: 20724
Columns: 34

Saved district-level file: data/district_final_risk_score.csv
Rows: 1980
Columns: 29

Saved district-level file: data/district_final_risk_score.csv
Rows: 1980
Columns: 29


In [7]:
import pandas as pd

df = pd.read_csv("data/district_final_risk_score.csv")

print(df["timeperiod"].min())
print(df["timeperiod"].max())

print(
    sorted(df["timeperiod"].unique())[-10:]
)

2021_01
2026_06
['2025_09', '2025_10', '2025_11', '2025_12', '2026_01', '2026_02', '2026_03', '2026_04', '2026_05', '2026_06']


In [ ]:
#testing (visualization)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from ipywidgets import interact, Dropdown

# =====================================================
# INPUT FILES
# =====================================================
GEOJSON = "../Maps/Geojson/district.geojson"
CSV = "data/district_final_risk_score.csv"

# =====================================================
# LOAD DATA
# =====================================================
gdf = gpd.read_file(GEOJSON)
risk_df = pd.read_csv(CSV)

# =====================================================
# CLEAN JOIN KEYS
# =====================================================
gdf["object_id"] = gdf["object_id"].astype(str).str.strip()
risk_df["object-id"] = risk_df["object-id"].astype(str).str.strip()

# =====================================================
# AVAILABLE TIME PERIODS
# =====================================================
timeperiods = sorted(risk_df["timeperiod"].dropna().unique())

# =====================================================
# NUMERIC RISK COLUMNS ONLY
# =====================================================
exclude = {
    "object-id",
    "timeperiod",
}

plot_cols = [
    c for c in risk_df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(risk_df[c])
]

print("Columns available:")
print(plot_cols)

# =====================================================
# FIXED RISK COLORMAP
# =====================================================
cmap = ListedColormap([
    "#3E4F92",  # 1 Very Low
    "#65A4BD",  # 2 Low
    "#FFED6E",  # 3 Moderate
    "#FB8C35",  # 4 High
    "#D41505",  # 5 Very High
])

# =====================================================
# PLOT FUNCTION
# =====================================================
def plot_map(column, timeperiod):

    # Filter selected time period
    df_tp = risk_df.loc[risk_df["timeperiod"] == timeperiod].copy()

    # Merge
    gdf_plot = gdf.merge(
        df_tp,
        left_on="object_id",
        right_on="object-id",
        how="left"
    )

    # Diagnostics
    print(f"\nTimeperiod: {timeperiod}")
    print(f"Column: {column}")
    print(gdf_plot[column].value_counts(dropna=False).sort_index())

    fig, ax = plt.subplots(figsize=(10, 10))

    gdf_plot.plot(
        column=column,
        cmap=cmap,
        vmin=1,
        vmax=5,
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        ax=ax,
        missing_kwds={
            "color": "lightgrey",
            "label": "No Data"
        }
    )

    legend = [
        Patch(facecolor="#3E4F92", edgecolor="black", label="1 - Very Low"),
        Patch(facecolor="#65A4BD", edgecolor="black", label="2 - Low"),
        Patch(facecolor="#FFED6E", edgecolor="black", label="3 - Moderate"),
        Patch(facecolor="#FB8C35", edgecolor="black", label="4 - High"),
        Patch(facecolor="#D41505", edgecolor="black", label="5 - Very High"),
    ]

    ax.legend(
        handles=legend,
        title="Risk Level",
        loc="lower left",
        frameon=True
    )

    ax.set_title(f"{column} ({timeperiod})", fontsize=15, weight="bold")
    ax.set_axis_off()

    plt.tight_layout()
    plt.show()

# =====================================================
# INTERACTIVE DROPDOWNS
# =====================================================
interact(
    plot_map,
    column=Dropdown(
        options=plot_cols,
        value=plot_cols[0],
        description="Column:",
        layout={"width": "350px"},
    ),
    timeperiod=Dropdown(
        options=timeperiods,
        value=timeperiods[0],
        description="Time:",
        layout={"width": "220px"},
    ),
);

Columns available:
['health-centres-count', 'total-tender-awarded-value-fy-cumsum', 'total-tender-awarded-value', 'heat-days-score', 'sum-aged-population', 'sum-young-population', 'sum-population', 'avg-electricity', 'piped-hhds-pct', 'nosanitation-hhds-pct', 'total-hhd', 'workers-affected-pct', 'women-sugar', 'men-sugar', 'women-bp', 'men-bp', 'pct-ncd', 'heat-hazard', 'exposure', 'vulnerability', 'government-response', 'topsis-score', 'heat-risk-score', 'land-surface-temperature']


interactive(children=(Dropdown(description='Column:', layout=Layout(width='350px'), options=('health-centres-c…